# HydroSense · All-variable predictive maintenance pilot

Train the **same TFT + LSTM + Isolation Forest models** used by the app, on any numeric tag names and tag counts. Every tag in your unit is forecast. Export a model for local PI inference and a chronological holdout report for engineering review.

For Colab: choose **Runtime → Change runtime type → GPU**. Only upload data your organization permits in Google Colab. You can run this notebook on an approved local Jupyter host instead. No PI connection or credentials are needed for training.

Model semantics: a past multivariate window predicts future values for every tag. Current observations are compared with a forecast issued earlier. The dataset must have time-aligned rows; windows crossing missing values or cadence gaps are excluded.

In [ ]:
from pathlib import Path
import subprocess, sys
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo = Path('/content/hydrosense')
    if not repo.exists():
        subprocess.run(['git', 'clone', 'https://github.com/bunjopolo/hydrosense.git', str(repo)], check=True)
else:
    repo = Path.cwd()
    if repo.name == 'notebooks':
        repo = repo.parent
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo) + '[notebooks]'], check=True)
sys.path.insert(0, str(repo / 'src'))

## 1. Choose training data

Upload one CSV or `.xlsx` workbook in Colab, or set `DATA_PATH` to a local file. Select the correct timestamp column, timezone for local timestamps, and Excel sheet. All numeric columns other than the timestamp become model tags automatically. Remove numeric record IDs or metadata columns from the source file if they are not physical variables you want forecast.

In [ ]:
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one training file.')
    DATA_PATH = next(iter(uploaded))
else:
    DATA_PATH = '/path/to/healthy-unit-history.csv'

TIMESTAMP_COLUMN = 'timestamp'
SOURCE_TIMEZONE = 'UTC'  # e.g. America/Vancouver for local Excel timestamps
EXCEL_SHEET = ''        # blank selects the first sheet
UNIT_NAME = 'Pilot hydro unit'
HEALTHY_HISTORY_CONFIRMED = False  # set True only after reviewing faults and maintenance

## 2. Configure history, horizon and training

`lookback` and `horizon` are numbers of samples. For one-minute data, 72 / 12 means 72 minutes of history and 12 minutes ahead. There is no input/target split: all variables are predicted. CUDA is selected if available; the app can load the resulting CPU-portable model.

The earliest 60% fits the models; the next 20% calibrates residual thresholds; the final 20% evaluates forecasts. Do not tune repeatedly against the held-out test results and call that an independent evaluation. Reserve a separate prospective period for the SmartSignal comparison.

In [ ]:
import json, torch
from hydrosense.data import read_file
from hydrosense.model import config_for_frame, train
config = json.loads((repo / 'configs/hydro/assets.json').read_text())
config.update(timestamp_column=TIMESTAMP_COLUMN, source_timezone=SOURCE_TIMEZONE,
              lookback=72, horizon=12, epochs=30, batch_size=128, hidden_size=32,
              device='cuda' if torch.cuda.is_available() else 'cpu')
frame = read_file(DATA_PATH, config, EXCEL_SHEET)
config = config_for_frame(frame, config, UNIT_NAME)
print('Device:', config['device'], '| rows:', len(frame), '| tags:', len(config['assets'][0]['tags']))
print('Tags:', config['assets'][0]['tags'])
display(frame.head())

In [ ]:
if not HEALTHY_HISTORY_CONFIRMED:
    raise ValueError('Review training history and set HEALTHY_HISTORY_CONFIRMED = True before training.')
bundle = train(frame, config, progress=print)
print('Completed:', bundle['created_at'])

## 3. Review every tag’s holdout results

MAE uses the original engineering units across the full forecast horizon. Compare TFT MAE with the simple last-observed-value baseline. A worse TFT result is evidence to investigate settings/data suitability—not a result to hide. These metrics do not establish maintenance fault sensitivity or false-alarm rates.

In [ ]:
import pandas as pd
rows = []
for unit in bundle['reports']:
    for tag in unit['tags']:
        rows.append({'unit': unit['name'], **tag})
metrics = pd.DataFrame(rows)
display(metrics)
ax = metrics.set_index('tag')[['mae', 'baseline_mae']].plot.bar(figsize=(max(10, len(metrics) * .5), 4))
ax.set_ylabel('MAE in each tag’s source units (scales differ)')
ax.set_title('Held-out TFT vs last-value baseline')

## 4. Export for local inference

The export contains fitted transforms, model weights, the tag schema, configuration, training timestamp and validation metrics. It contains learned information about your training data; keep it private. The repository does not track model artifacts.

Joblib is executable serialization. Load only a model you trained or trust. On the local computer, stop HydroSense, copy the model there using an approved transfer method, install the same project version, and run:

```text
python -m hydrosense.cli activate --model hydro-model.joblib --trust-model
```

Restart HydroSense. Set the PI polling interval to the historical sample cadence. The app warms up from incoming PI observations and then forecasts all variables. Use **Export pilot predictions** to obtain timestamped actuals, prior forecasts and individual deviation states for comparison with SmartSignal.

In [ ]:
import joblib, zipfile
output = repo / 'artifacts' / 'notebook-export'
output.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, output / 'hydro-model.joblib')
(output / 'config.json').write_text(json.dumps(bundle['config'], indent=2))
metrics.to_csv(output / 'holdout-metrics.csv', index=False)
# Record the training software versions for reproducibility.
import importlib.metadata as metadata
versions = {name: metadata.version(name) for name in ['torch', 'lightning', 'pytorch-forecasting', 'scikit-learn', 'numpy', 'pandas', 'joblib']}
(output / 'versions.json').write_text(json.dumps(versions, indent=2))
archive = output / 'hydrosense-trained-model.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ['hydro-model.joblib', 'config.json', 'holdout-metrics.csv', 'versions.json']:
        z.write(output / name, name)
print('Saved:', archive)
if IN_COLAB:
    files.download(str(archive))

## Optional: replay a separate file

Use a different, engineer-reviewed period with known events. Neither the model nor its thresholds are refit during replay. Load that file in the app for the full all-variable dashboard, or use:

```text
python -m hydrosense.cli score --data pilot-period.csv --model hydro-model.joblib --output pilot-predictions.jsonl
```

For the side-by-side pilot, align both systems to UTC, identical units and sampling cadence; record each system’s issue time, first deviation, persistence and recovery. Compare against engineer-labelled events, including false alarms and time to detect. HydroSense does not yet import SmartSignal output or compute a combined comparison automatically.